# S1 · Arquitectura Big Data: Heladas en Juliaca
**Proyecto Sello · Equipo 10 · Nilver Salcca Aquino · Big Data UPeU 2026-2**

**Pregunta central:** ¿con qué anticipación y precisión se puede predecir la helada meteorológica (temperatura mínima ≤ 0 °C, según SENAMHI) en Juliaca, a partir de su temperatura, punto de rocío, humedad y nubosidad?

**Corte de datos:** todo el histórico llega como máximo hasta el **01-Oct-2026**.

## 1. Verificar el entorno
### 1.1 Crear la SparkSession
Confirma que Spark funciona dentro del contenedor `lambda26-pyspark`.

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("s1-sello-heladas")
    .master("local[*]")
    .config("spark.ui.port", "4040")
    .getOrCreate()
)
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/20 01:20:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 2. Configuración de la descarga
### 2.1 Rutas, fechas de corte y clave de la API
La clave se lee del archivo `/opt/u1/.env` (nunca se escribe en el notebook ni se imprime).

In [2]:
import os, csv, json, time, urllib.request, urllib.error, urllib.parse
from datetime import date, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed

RUTA_BASE = "/opt/u1"
RUTA_ENV  = f"{RUTA_BASE}/.env"
RUTA_HIST = f"{RUTA_BASE}/s01-arquitectura/data/historico"
os.makedirs(RUTA_HIST, exist_ok=True)

INICIO = date(2010, 1, 1)   # se ajusta en la sección 3 si la API no llega a 2010
FIN    = date(2026, 10, 1)  # CORTE: nunca se descarga nada posterior
CIUDAD = "Juliaca,Peru"
CAMPOS = ["ciudad", "hora", "temperatura", "puntoRocio", "humedad", "nubosidad", "presion", "precipitacion"]

def leer_clave(ruta):
    for linea in open(ruta, encoding="utf-8-sig"):
        if linea.startswith("WEATHERAPI_KEY="):
            return linea.split("=", 1)[1].strip()
    raise RuntimeError("No se encontró WEATHERAPI_KEY en " + ruta)

CLAVE = leer_clave(RUTA_ENV)
print("Clave cargada. Longitud:", len(CLAVE))
print("Rango solicitado:", INICIO, "->", FIN)

Clave cargada. Longitud: 30
Rango solicitado: 2010-01-01 -> 2026-10-01


### 2.2 Función que descarga un día (24 lecturas horarias)

In [3]:
def descargar_dia(d, reintentos=3):
    params = urllib.parse.urlencode({"key": CLAVE, "q": CIUDAD, "dt": d.isoformat()})
    url = "https://api.weatherapi.com/v1/history.json?" + params
    for intento in range(1, reintentos + 1):
        try:
            with urllib.request.urlopen(url, timeout=30) as r:
                datos = json.loads(r.read().decode("utf-8"))
            horas = datos["forecast"]["forecastday"][0]["hour"]
            return [("Juliaca", h["time"], h["temp_c"], h["dewpoint_c"], h["humidity"],
                     h["cloud"], h["pressure_mb"], h["precip_mm"]) for h in horas]
        except urllib.error.HTTPError as e:
            cuerpo = e.read().decode("utf-8", "ignore")[:200]
            if e.code in (401, 403):
                raise RuntimeError(f"HTTP {e.code}: la clave no tiene acceso. {cuerpo}")
            if e.code == 400:
                return None            # fecha sin datos o fuera de plan
            time.sleep(2 * intento)
        except Exception:
            time.sleep(2 * intento)
    return None

## 3. Prueba de disponibilidad
### 3.1 ¿Hasta qué año atrás llega la clave?
Busca el primer 1 de enero que la API entrega, empezando en 2010, y ajusta `INICIO`.

In [4]:
INICIO_REAL = None
for anio in range(2010, 2027):
    prueba = descargar_dia(date(anio, 1, 1))
    estado = f"{len(prueba)} horas" if prueba else "sin datos"
    print(f"{anio}-01-01 -> {estado}")
    if prueba:
        INICIO_REAL = date(anio, 1, 1)
        break

if INICIO_REAL is None:
    raise RuntimeError("La API no entregó datos en ningún año desde 2010.")
INICIO = INICIO_REAL
print("\nINICIO definitivo:", INICIO, "| FIN (corte):", FIN)
print("Días a descargar:", (FIN - INICIO).days + 1)

2010-01-01 -> 24 horas

INICIO definitivo: 2010-01-01 | FIN (corte): 2026-10-01
Días a descargar: 6118


## 4. Descarga del histórico (por años)
### 4.1 Una descarga por año, con reanudación
Cada año se guarda en su propio CSV. Si el notebook se interrumpe, al volver a ejecutar **salta los años ya descargados**.

In [5]:
def descargar_anio(anio, hilos=6):
    archivo = f"{RUTA_HIST}/clima_juliaca_{anio}.csv"
    if os.path.exists(archivo):
        return "ya existía", 0
    ini = max(date(anio, 1, 1), INICIO)
    fin = min(date(anio, 12, 31), FIN)
    dias = [ini + timedelta(n) for n in range((fin - ini).days + 1)]
    filas, fallidos = {}, []
    with ThreadPoolExecutor(max_workers=hilos) as ex:
        futuros = {ex.submit(descargar_dia, d): d for d in dias}
        for f in as_completed(futuros):
            d = futuros[f]
            res = f.result()
            if res is None:
                fallidos.append(d)
            else:
                filas[d] = res
    if fallidos:
        return f"INCOMPLETO ({len(fallidos)} días fallidos)", len(fallidos)
    with open(archivo, "w", newline="", encoding="utf-8") as fh:
        w = csv.writer(fh)
        w.writerow(CAMPOS)
        for d in sorted(filas):
            w.writerows(filas[d])
    return f"{len(filas)} días", 0

### 4.2 Ejecutar la descarga
Puede tardar varios minutos. Muestra el avance año por año.

In [6]:
t0 = time.time()
resumen = []
for anio in range(INICIO.year, FIN.year + 1):
    estado, fallos = descargar_anio(anio)
    resumen.append((anio, estado, fallos))
    print(f"{anio}: {estado}   [{time.time() - t0:5.0f} s]")

pendientes = [r for r in resumen if r[2] > 0]
print("\nAños con problemas:", pendientes if pendientes else "ninguno")

2010: 365 días   [   30 s]
2011: 365 días   [   60 s]
2012: 366 días   [   93 s]
2013: 365 días   [  124 s]
2014: 365 días   [  153 s]
2015: 365 días   [  184 s]
2016: 366 días   [  214 s]
2017: 365 días   [  243 s]
2018: 365 días   [  273 s]
2019: 365 días   [  303 s]
2020: 366 días   [  332 s]
2021: 365 días   [  361 s]
2022: 365 días   [  400 s]
2023: 365 días   [  431 s]
2024: 366 días   [  460 s]
2025: 365 días   [  490 s]
2026: 274 días   [  512 s]

Años con problemas: ninguno


## 5. Verificación con Spark
### 5.1 Leer todo el histórico con esquema explícito

In [7]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

esquema = StructType([
    StructField("ciudad",       StringType(),  True),
    StructField("hora",         StringType(),  True),
    StructField("temperatura",  DoubleType(),  True),
    StructField("puntoRocio",   DoubleType(),  True),
    StructField("humedad",      IntegerType(), True),
    StructField("nubosidad",    IntegerType(), True),
    StructField("presion",      DoubleType(),  True),
    StructField("precipitacion",DoubleType(),  True),
])

df = (spark.read.csv(f"{RUTA_HIST}/*.csv", header=True, schema=esquema)
      .withColumn("hora", F.to_timestamp("hora")))

df.printSchema()
df.show(5, truncate=False)

26/11/20 01:31:05 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: /opt/u1/s01-arquitectura/data/historico/*.csv.
java.io.FileNotFoundException: File /opt/u1/s01-arquitectura/data/historico/*.csv does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:394)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:210

root
 |-- ciudad: string (nullable = true)
 |-- hora: timestamp (nullable = true)
 |-- temperatura: double (nullable = true)
 |-- puntoRocio: double (nullable = true)
 |-- humedad: integer (nullable = true)
 |-- nubosidad: integer (nullable = true)
 |-- presion: double (nullable = true)
 |-- precipitacion: double (nullable = true)

+-------+-------------------+-----------+----------+-------+---------+-------+-------------+
|ciudad |hora               |temperatura|puntoRocio|humedad|nubosidad|presion|precipitacion|
+-------+-------------------+-----------+----------+-------+---------+-------+-------------+
|Juliaca|2016-01-01 00:00:00|4.1        |-0.8      |70     |4        |1017.0 |0.0          |
|Juliaca|2016-01-01 01:00:00|3.7        |-1.5      |69     |3        |1017.0 |0.0          |
|Juliaca|2016-01-01 02:00:00|3.2        |-2.1      |68     |3        |1017.0 |0.0          |
|Juliaca|2016-01-01 03:00:00|2.8        |-2.8      |66     |2        |1018.0 |0.0          |
|Juliaca|2016-0

### 5.2 Cobertura y corte de fechas

In [8]:
total = df.count()
esperado = ((FIN - INICIO).days + 1) * 24
print("Filas leídas :", total)
print("Filas esperadas:", esperado)

df.agg(F.min("hora").alias("primera"), F.max("hora").alias("ultima")).show(truncate=False)

print("Filas por año:")
df.groupBy(F.year("hora").alias("anio")).count().orderBy("anio").show(30)

posteriores = df.filter(F.col("hora") > F.lit("2026-10-01 23:59:59")).count()
nulos_temp = df.filter(F.col("temperatura").isNull()).count()
print("Filas posteriores al 01-Oct-2026:", posteriores)
print("Nulos en temperatura:", nulos_temp)

assert posteriores == 0, "Hay datos posteriores al corte del 01-Oct-2026"
print("\nCorte verificado: no hay datos posteriores al 01-Oct-2026.")

Filas leídas : 146832
Filas esperadas: 146832
+-------------------+-------------------+
|primera            |ultima             |
+-------------------+-------------------+
|2010-01-01 00:00:00|2026-10-01 23:00:00|
+-------------------+-------------------+

Filas por año:
+----+-----+
|anio|count|
+----+-----+
|2010| 8760|
|2011| 8760|
|2012| 8784|
|2013| 8760|
|2014| 8760|
|2015| 8760|
|2016| 8784|
|2017| 8760|
|2018| 8760|
|2019| 8760|
|2020| 8784|
|2021| 8760|
|2022| 8760|
|2023| 8760|
|2024| 8784|
|2025| 8760|
|2026| 6576|
+----+-----+

Filas posteriores al 01-Oct-2026: 0
Nulos en temperatura: 0

Corte verificado: no hay datos posteriores al 01-Oct-2026.


## 6. Spark UI

### 6.1 Ejecutar un job y verlo en Spark UI
Resume el histórico por año. Spark ejecuta un job que queda registrado en `http://localhost:4042`.


In [9]:
resumen = (
    df.groupBy(F.year("hora").alias("anio"))
      .agg(F.count("*").alias("lecturas"),
           F.round(F.avg("temperatura"), 2).alias("temp_media"),
           F.min("temperatura").alias("temp_min"))
      .orderBy("anio")
)
resumen.show(20)

print("Spark UI (en el contenedor):", spark.sparkContext.uiWebUrl)
print("Spark UI (en tu navegador): http://localhost:4042")

+----+--------+----------+--------+
|anio|lecturas|temp_media|temp_min|
+----+--------+----------+--------+
|2010|    8760|      8.87|    -5.3|
|2011|    8760|      7.94|    -4.4|
|2012|    8784|       8.0|    -3.8|
|2013|    8760|      8.24|    -4.3|
|2014|    8760|      8.31|    -4.3|
|2015|    8760|      7.84|    -5.4|
|2016|    8784|      8.53|    -4.2|
|2017|    8760|       8.3|    -2.5|
|2018|    8760|      8.14|    -2.7|
|2019|    8760|      8.56|    -4.0|
|2020|    8784|       9.4|    -3.0|
|2021|    8760|      8.98|    -2.4|
|2022|    8760|      8.86|    -4.3|
|2023|    8760|      9.23|    -3.2|
|2024|    8784|       9.6|    -3.9|
|2025|    8760|      8.84|    -1.8|
|2026|    6576|      8.33|    -5.2|
+----+--------+----------+--------+

Spark UI (en el contenedor): http://5e498c5b8c15:4040
Spark UI (en tu navegador): http://localhost:4042


## 7. Ecosistema del proyecto
### 7.1 Flujo base (3.3)

WeatherAPI -> Kafka -> Spark (Streaming + Batch) -> Parquet (Data Lake) -> Grafana

**¿Dónde nacen los eventos?**
En el modelo meteorológico de WeatherAPI para Juliaca: cada lectura trae temperatura, punto de rocío, humedad y nubosidad. En vivo se actualiza cada ~15 minutos; el histórico está disponible por hora.

**¿Qué componente ingesta los eventos?**
Kafka (propuesto) recibirá las lecturas en vivo. El histórico entra por archivos CSV al batch, igual que en lambda26, donde solo el pipeline en tiempo real lee de Kafka.

**¿Qué componente procesa los datos a escala?**
Spark: Structured Streaming para las lecturas en vivo y Spark batch para el histórico 2010 – 01-Oct-2026.

**¿Dónde se consumen los resultados?**
En Grafana (propuesto), en un tablero de heladas. Parquet es el almacenamiento analítico del batch.

## 8. Clasificación batch / streaming
### 8.1 ¿El caso requiere batch, streaming o ambos? (3.4)

**Respuesta: ambos.**

**Razones que piden BATCH**
1. Hay un histórico acumulado de 16 años: 146 832 lecturas horarias entre 2010-01-01 y 2026-10-01. Para saber en qué meses del año hay más heladas hay que procesar todos esos años juntos, en bloque.
2. El modelo de clasificación de U1 se entrena con ese histórico y se calcula una sola vez; no necesita reaccionar dato por dato.

**Razones que piden STREAMING**
1. Las lecturas del clima llegan de forma continua (en vivo se actualizan cada ~15 minutos) y la helada ocurre en horas, de madrugada.
2. La pregunta de U2 es de anticipación: ¿la temperatura de dentro de 1 hora será ≤ 0 °C? Un reporte diario llegaría demasiado tarde para que el agricultor cubra sus cultivos.

| Enfoque | Qué datos | Qué responde |
|---|---|---|
| Batch (U1) | Histórico por hora 2010 – 01-Oct-2026 | ¿En qué meses hay más heladas y qué tan bien se predicen? |
| Streaming (U2) | Lecturas en vivo | ¿Habrá helada en la próxima hora? |

## 9. Arquitectura seleccionada
### 9.1 Regla de decisión aplicada al caso (3.5)

**Regla (2.5.1):**
- Si el caso necesita histórico + tiempo real, entonces Lambda.
- Si todo el caso son eventos en tiempo real, entonces Kappa.

**Selección: Lambda.**

**Justificación:** el caso necesita las dos cosas. El histórico de 16 años (batch) sirve para saber en qué meses hay más heladas y entrenar el modelo de U1; las lecturas en vivo (streaming) sirven para avisar la temperatura de dentro de 1 hora en U2. Kappa se descarta porque el histórico por hora es una fuente batch separada del flujo en vivo.

**Desventaja asumida:** Lambda es más compleja (batch, speed y serving) y existe riesgo de deriva entre lo que calcula cada capa. Se mitiga usando la misma variable (helada: temperatura ≤ 0 °C) en las dos capas y mostrándolas juntas en un solo tablero.

## 10. Tecnologías y diagrama de flujo
### 10.1 Tecnologías propuestas (3.6)

| Etapa | Tecnología | Para qué |
|---|---|---|
| Fuente de datos | WeatherAPI | Lecturas de clima de Juliaca (en vivo y por hora) |
| Ingesta | Kafka (propuesto) + archivos CSV | Kafka para lo en vivo; CSV para el histórico |
| Procesamiento | Spark Structured Streaming y Spark batch | Streaming para U2, batch para U1 |
| Almacenamiento | Parquet (Data Lake) | Salida analítica del batch, particionada por año |
| Visualización | Grafana (propuesto) | Tablero de heladas con U1 y U2 |

### 10.2 Diagrama de flujo

```
WeatherAPI --(en vivo, ~15 min)--> Kafka --> Spark Streaming --> modelo +1 h ------> Grafana
WeatherAPI --(histórico por hora)--> CSV --> Spark Batch --> Parquet --> modelo clasificación --> Grafana
```